# 03 — Feature Engineering & Importance

Correlation analysis, collinearity, feature vs target, engineered features,
Track2Vec embeddings, and model feature importance.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from sklearn.inspection import permutation_importance
from sklearn.manifold import TSNE

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from etl.db import get_connection, init_schema
from paths import MODELS_DIR
from recommend.modules.classifiers import CLASSIFIER_FEATURES, CATEGORICAL_FEATURES
from recommend.modules.similarity import SIMILARITY_FEATURES
from recommend.preprocessing import build_feature_matrix
from utils.logging import configure_logging

configure_logging()
sns.set_theme(style="whitegrid", palette="muted")

conn = get_connection(read_only=False)
init_schema(conn)
corpus = build_feature_matrix(conn)
conn.close()

print(f"Corpus: {corpus.shape}")
print(f"SIMILARITY_FEATURES ({len(SIMILARITY_FEATURES)}): {SIMILARITY_FEATURES}")
print(f"CLASSIFIER_FEATURES ({len(CLASSIFIER_FEATURES)}): {CLASSIFIER_FEATURES}")

## 1. Feature Correlation Matrix

In [ ]:
# Correlations across SIMILARITY_FEATURES
available_sim = [f for f in SIMILARITY_FEATURES if f in corpus.columns]
corr_data = corpus.select(available_sim).to_pandas()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax,
    square=True, linewidths=0.5,
)
ax.set_title("Feature Correlation Matrix (SIMILARITY_FEATURES)")
plt.tight_layout()
plt.show()

### 1b. Full Correlation Matrix (incl. fave_score, popularity)

Extended correlation matrix including fave_score and popularity to surface
which audio features predict favourites.

In [ ]:
# Extended correlation heatmap: audio features + popularity + fave_score
AUDIO_FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
]
extra_cols = ["popularity", "fave_score"]
corr_cols = [c for c in AUDIO_FEATURES + extra_cols if c in corpus.columns]

full_corr = corpus.select(corr_cols).to_pandas().corr()
mask_full = np.triu(np.ones_like(full_corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    full_corr, mask=mask_full, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, linewidths=0.5,
    vmin=-1, vmax=1, ax=ax,
)
ax.set_title("Audio Feature Correlation Matrix (incl. popularity + fave_score)")
plt.tight_layout()
plt.show()

## 2. Collinearity Check

In [ ]:
# Flag pairs with |corr| > 0.9
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.9:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], round(val, 3)))

if high_corr_pairs:
    print("High collinearity pairs (|r| > 0.9):")
    for a, b, r in high_corr_pairs:
        print(f"  {a} ↔ {b}: {r}")
else:
    print("No feature pairs with |r| > 0.9")

## 3. Feature vs Target (per playlist)

In [ ]:
# Load ENOA to get playlist memberships
from paths import ARCHIVED_DIR

enoa_path = ARCHIVED_DIR / "enoa.csv"
if enoa_path.exists():
    enoa = pl.read_csv(enoa_path, null_values=["", "NA", "NaN"])
    playlist_names = enoa["playlist_name"].unique().sort().to_list()
    print(f"Available playlists: {playlist_names}")
else:
    print(f"ENOA not found at {enoa_path}")
    enoa = None

In [ ]:
# Pick a playlist to analyze (change this)
TARGET_PLAYLIST = playlist_names[0] if enoa is not None else None

if enoa is not None and TARGET_PLAYLIST:
    playlist_ids = set(
        enoa.filter(pl.col("playlist_name") == TARGET_PLAYLIST)["id"]
        .cast(pl.Utf8).to_list()
    )
    corpus_ids = corpus["id"].cast(pl.Utf8).to_list()
    labels = np.array([1 if tid in playlist_ids else 0 for tid in corpus_ids])

    print(f"Playlist: {TARGET_PLAYLIST}")
    print(f"Positives: {labels.sum()}, Negatives: {(labels == 0).sum()}")

    # Violin plots for top features
    plot_features = available_sim[:9]  # audio features
    corpus_pd = corpus.select(plot_features).to_pandas()
    corpus_pd["label"] = labels

    fig, axes = plt.subplots(3, 3, figsize=(14, 10))
    axes = axes.flatten()
    for i, feat in enumerate(plot_features):
        sns.violinplot(data=corpus_pd, x="label", y=feat, ax=axes[i], cut=0)
        axes[i].set_title(feat)
        axes[i].set_xlabel("")

    plt.suptitle(f"Feature Distributions: {TARGET_PLAYLIST} (1=member, 0=not)", fontsize=13)
    plt.tight_layout()
    plt.show()

### 3b. Fave Score Analysis

Avg fave score by genre group + scatter regression of fave_score vs key audio features.

In [ ]:
# Fave score by genre group + regression vs audio features
if "fave_score" in corpus.columns and "gen_4" in corpus.columns:
    fave_data = corpus.filter(pl.col("fave_score").is_not_null() & pl.col("gen_4").is_not_null())

    # Avg fave score by gen_4
    fave_by_genre = (
        fave_data.group_by("gen_4")
        .agg(pl.col("fave_score").mean().alias("avg_fave"))
        .sort("avg_fave")
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart
    axes[0].barh(fave_by_genre["gen_4"].to_list(), fave_by_genre["avg_fave"].to_list())
    axes[0].set_title("Avg fave score by gen_4")
    axes[0].set_xlabel("Mean fave_score")

    # Distribution
    fave_vals = fave_data["fave_score"].to_numpy()
    axes[1].hist(fave_vals, bins=30, edgecolor="white", alpha=0.8)
    axes[1].set_title("Fave score distribution")
    axes[1].set_xlabel("fave_score")
    plt.tight_layout()
    plt.show()

    # Scatter regression: fave_score vs key audio features
    reg_features = ["danceability", "energy", "valence", "acousticness", "instrumentalness"]
    available_reg = [f for f in reg_features if f in fave_data.columns]

    if available_reg:
        fave_pd = fave_data.select(available_reg + ["fave_score"]).to_pandas()
        fig, axes = plt.subplots(1, len(available_reg), figsize=(4 * len(available_reg), 3.5))
        if len(available_reg) == 1:
            axes = [axes]
        for ax, feat in zip(axes, available_reg):
            sns.regplot(
                data=fave_pd, x=feat, y="fave_score",
                scatter_kws={"alpha": 0.2, "s": 8},
                line_kws={"color": "red"},
                ax=ax,
            )
            ax.set_title(feat)
        plt.suptitle("Fave Score vs Audio Features", y=1.02, fontsize=13)
        plt.tight_layout()
        plt.show()
else:
    print("fave_score or gen_4 column not available")

## 4. Engineered Features Inspection

In [ ]:
engineered = [
    "year_normalized", "years_since_release", "duration_ms_normalized",
    "playlist_diversity", "fave_score", "n_playlists",
    "artist_enoa_top", "artist_enoa_left",
]
available_eng = [f for f in engineered if f in corpus.columns]

ncols = 3
nrows = (len(available_eng) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
axes = axes.flatten()

for i, feat in enumerate(available_eng):
    values = corpus[feat].drop_nulls().to_numpy()
    axes[i].hist(values, bins=50, edgecolor="white", alpha=0.8)
    axes[i].set_title(f"{feat} (μ={np.mean(values):.3f})")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Engineered Feature Distributions", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Propagated playlist profile features
pp_cols = [c for c in corpus.columns if c.startswith("pp_")]
if pp_cols:
    print(f"Propagated features ({len(pp_cols)}): {pp_cols}")
    corpus.select(pp_cols).describe()
else:
    print("No pp_* columns found")

## 5. Track2Vec Embeddings (t-SNE)

In [ ]:
t2v_cols = [c for c in corpus.columns if c.startswith("t2v_")]

if t2v_cols and len(t2v_cols) > 0:
    # Filter to tracks with non-zero embeddings
    has_emb = corpus.filter(
        pl.any_horizontal([pl.col(c) != 0 for c in t2v_cols])
    )
    print(f"Tracks with embeddings: {len(has_emb):,}")

    # Subsample for t-SNE performance
    TSNE_SAMPLE = min(3000, len(has_emb))
    sample = has_emb.sample(n=TSNE_SAMPLE, seed=42)

    X_emb = sample.select(t2v_cols).to_numpy()
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    coords = tsne.fit_transform(X_emb)

    # Color by gen_4 if available
    fig, ax = plt.subplots(figsize=(12, 10))
    if "gen_4" in sample.columns:
        gen4_vals = sample["gen_4"].fill_null("unknown").to_list()
        unique_genres = sorted(set(gen4_vals))
        color_map = {g: i for i, g in enumerate(unique_genres)}
        colors = [color_map[g] for g in gen4_vals]
        scatter = ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=5, alpha=0.5, cmap="tab10")
        # Legend
        handles = [plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=plt.cm.tab10(color_map[g] / len(unique_genres)),
                   markersize=8, label=g) for g in unique_genres]
        ax.legend(handles=handles, title="gen_4")
    else:
        ax.scatter(coords[:, 0], coords[:, 1], s=5, alpha=0.5)

    ax.set_title(f"Track2Vec t-SNE (n={TSNE_SAMPLE})")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    plt.tight_layout()
    plt.show()
else:
    print("No Track2Vec columns found")

## 6. LightGBM Feature Importance

In [ ]:
from recommend.modules.classifiers import ALL_DECADES, ALL_GEN4, load_classifier

# Feature names for LightGBM (numeric + one-hot encoded categoricals)
lgbm_feature_names = CLASSIFIER_FEATURES + [f"decade_{d}" for d in ALL_DECADES] + [f"gen4_{g}" for g in ALL_GEN4]

# Load a classifier — pick one that exists
classifier_pkls = sorted(MODELS_DIR.glob("classifier_*.pkl"))
print(f"Available classifiers: {len(classifier_pkls)}")

if classifier_pkls:
    # Extract importances from up to 5 classifiers
    all_importances = {}
    for pkl_path in classifier_pkls[:5]:
        slug = pkl_path.stem.replace("classifier_", "")
        pipeline = joblib.load(pkl_path)

        # Navigate pipeline to get the base estimator
        clf = pipeline.named_steps.get("classifier", pipeline)
        # CalibratedClassifierCV wraps the actual estimator
        if hasattr(clf, "estimator") and hasattr(clf.estimator, "feature_importances_"):
            importances = clf.estimator.feature_importances_
        elif hasattr(clf, "calibrated_classifiers_"):
            # Average across CV folds
            fold_imps = [c.estimator.feature_importances_ for c in clf.calibrated_classifiers_ if hasattr(c.estimator, "feature_importances_")]
            if fold_imps:
                importances = np.mean(fold_imps, axis=0)
            else:
                continue
        elif hasattr(clf, "feature_importances_"):
            importances = clf.feature_importances_
        else:
            print(f"  {slug}: no feature_importances_ found")
            continue

        all_importances[slug] = importances
        print(f"  {slug}: {len(importances)} features")

    if all_importances:
        # Average importance across playlists
        n_features = len(next(iter(all_importances.values())))
        mean_imp = np.mean(list(all_importances.values()), axis=0)
        feat_names = lgbm_feature_names[:n_features] if n_features <= len(lgbm_feature_names) else [f"f{i}" for i in range(n_features)]

        # Sort by importance
        sorted_idx = np.argsort(mean_imp)[::-1]

        fig, ax = plt.subplots(figsize=(10, 8))
        top_n = min(20, len(sorted_idx))
        ax.barh(
            [feat_names[i] for i in sorted_idx[:top_n]][::-1],
            [mean_imp[i] for i in sorted_idx[:top_n]][::-1],
        )
        ax.set_xlabel("Mean Importance (across playlists)")
        ax.set_title(f"LightGBM Feature Importance (top {top_n}, averaged over {len(all_importances)} playlists)")
        plt.tight_layout()
        plt.show()
else:
    print("No classifier artifacts found — run train.py first")

## 7. Per-Playlist Importance Comparison

In [ ]:
if all_importances and len(all_importances) > 1:
    # Heatmap: playlists × features (top 15 features by mean importance)
    top_feat_idx = sorted_idx[:15]
    top_feat_names = [feat_names[i] for i in top_feat_idx]

    imp_matrix = np.array([all_importances[slug][top_feat_idx] for slug in all_importances])

    fig, ax = plt.subplots(figsize=(12, max(4, len(all_importances) * 0.6)))
    sns.heatmap(
        imp_matrix,
        xticklabels=top_feat_names,
        yticklabels=list(all_importances.keys()),
        cmap="YlOrRd", annot=True, fmt=".0f", ax=ax,
    )
    ax.set_title("Feature Importance per Playlist (top 15 features)")
    plt.tight_layout()
    plt.show()

## 8. Permutation Importance (optional, slow)

In [ ]:
# Uncomment to run — requires a fitted pipeline + test data
# This is slower but gives a more robust importance signal.
#
# from recommend.modules.classifiers import build_rerank_features, train_playlist_classifier
#
# if enoa is not None and TARGET_PLAYLIST:
#     pipeline, metrics = train_playlist_classifier(
#         corpus=corpus,
#         playlist_track_ids=playlist_ids,
#         scaler=joblib.load(MODELS_DIR / "scaler_corpus.pkl"),
#         model_type="lightgbm",
#     )
#     # Build test features
#     X_test = ...  # would need train_test_split output
#     perm_imp = permutation_importance(pipeline, X_test, y_test, n_repeats=10, random_state=42)
#     # Plot perm_imp.importances_mean
print("Permutation importance cell — uncomment and adapt to run")